In [13]:
import os
import sys
import pandas as pd
from PIL import Image
from transformers import AutoProcessor, Gemma3ForConditionalGeneration
import torch
import time
from datetime import datetime, timezone


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT = "/kaggle/input/datasets/yanncohen/ebbinghaus-illusion-benchmark"
WORK  = "/kaggle/working"
os.makedirs(f"{WORK}/images_eval", exist_ok=True)
os.makedirs(f"{WORK}/output",      exist_ok=True)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
# ── Phase 1: Load trials ──────────────────────────────────────────────────────
trials  = pd.read_csv(f"{INPUT}/data/trials.csv")
prompts = pd.read_csv(f"{INPUT}/data/prompts.csv")
print(f"Loaded {len(trials)} trials")

In [ ]:
# ── Phase 2: Strip ground truth from filenames ────────────────────────────────
sys.path.insert(0, f"{INPUT}/py/src")
from strip_answer import strip_answer_from_images

trials["file_path"] = trials["file_path"].apply(
    lambda p: f"{INPUT}/images/{os.path.basename(p)}"
)
trials = strip_answer_from_images(trials, output_dir=f"{WORK}/images_eval")

In [ ]:
# ── Phase 3: Load Gemma 3 (multimodal) ────────────────────────────────────────
model_id  = "google/gemma-3-4b-it"
processor = AutoProcessor.from_pretrained(model_id)
model     = Gemma3ForConditionalGeneration.from_pretrained(
                model_id, device_map="cuda", torch_dtype=torch.bfloat16
            )


In [ ]:
# ── Phase 4: Evaluation loop ──────────────────────────────────────────────────
def get_directions(orientation):
    if orientation == "horizontal":
        return "left", "right"
    elif orientation == "vertical":
        return "bottom", "top"
    elif orientation == "diagonal":
        return "bottom-left", "top-right"
    else:
        raise ValueError(f"Unknown orientation: {orientation}")

def parse_response(raw: str, direction_a: str, direction_b: str, response_format: str) -> str:
    """Extract the answer token from raw response."""
    if response_format == "free_text":
        # Extract answer after "ANSWER:" marker
        if "answer:" in raw:
            raw = raw.split("answer:")[-1].strip()
    if direction_a in raw:
        return direction_a
    if direction_b in raw:
        return direction_b
    if "equal" in raw:
        return "equal"
    return "unknown"

results = []

for _, prompt_row in prompts.iterrows():
    prompt_id       = prompt_row["prompt_id"]
    prompt_template = prompt_row["user_prompt_template"]
    response_format = prompt_row["response_format"]

    for _, row in trials.iterrows():
        direction_a, direction_b = get_directions(row["orientation"])

        raw_text = prompt_template.format(
            direction_a=direction_a,
            direction_b=direction_b,
            test_a_shape=row["test_a_shape"],
            test_b_shape=row["test_b_shape"]
        )

        messages = [
            {"role": "user", "content": [{"type": "image"}, {"type": "text", "text": raw_text}]}
        ]

        chat_prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
        image  = Image.open(row["file_path"])
        inputs = processor(
            images=image,
            text=chat_prompt,
            return_tensors="pt"
        ).to(model.device)

        t0 = time.time()
        with torch.no_grad():
            output = model.generate(**inputs, max_new_tokens=50)  # more tokens for free_text
        latency_ms = round((time.time() - t0) * 1000)

        input_len = inputs["input_ids"].shape[-1]
        response  = processor.decode(output[0][input_len:], skip_special_tokens=True).strip().lower()
        parsed    = parse_response(response, direction_a, direction_b, response_format)

        expected_map = {"a": direction_a, "b": direction_b, "equal": "equal"}
        expected     = expected_map[row["true_larger"]]

        results.append({
            "eval_id":             len(results) + 1,
            "trial_id":            row["trial_id"],
            "prompt_id":           prompt_id,
            "provider":            "google",
            "model":               "gemma-3-4b-it",
            "model_version":       "NA",
            "temperature":         0,
            "max_tokens":          50,
            "response_larger":     "a" if parsed == direction_a else ("b" if parsed == direction_b else parsed),
            "response_confidence": "NA",
            "raw_response":        response,
            "latency_ms":          latency_ms,
            "timestamp":           datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "error":               "NA"
        })

print(f"Total evals: {len(results)}")  # should be 80 (20 trials × 4 prompts × 1 model)


In [ ]:
# ── Phase 5: Save & summarise ─────────────────────────────────────────────────
evals = pd.DataFrame(results)
evals.to_csv(f"{WORK}/output/py-hg-evals.csv", index=False)

print(f"Saved {len(evals)} evals to py-hg-evals.csv")
print(evals.groupby("prompt_id")["response_larger"].value_counts())